1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path('/kaggle/working')
WAN_GP_DIR = ROOT / 'WanGP'

'''
Select the model precision and GPU arrangement.
選擇模型精度與 GPU 方案：int8_single 單卡 INT8；bf16_single 單卡 BF16；bf16_dual 雙卡 BF16。
'''
MODEL_MODE = 'bf16_dual'  # 'int8_single', 'bf16_single', 'bf16_dual'
GPU_ID = 0  # 0、1（須與 MODEL_MODE 相容）

'''
Set the GPU 0 transformer preload memory budget in GiB.
設定 GPU 0 Transformer 權重預載記憶體預算（GiB）。
'''
GPU0_MODEL_GIB = 14.0  # 14.0、13.5，依 GPU 記憶體調整
CKPT = Path('/tmp/qwen21_ckpts')
HF_CACHE_DIR = Path('/tmp/qwen21_hf')

'''
Apply the selected model-cache directory to Hugging Face downloads.
讓後續 Hugging Face 模型下載確實使用第一格所設定的快取路徑。
'''
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
NEGATIVE_PROMPT = None  # None（不指定），或改成字串
REFERENCE_SIZE = 512  # 例如 512、768，值越大通常越占用 GPU 記憶體
OUTPUT_NAME = 'qwen21_output.png'
SAMPLER = 'auto'  # 'auto' 或模型支援的採樣器
CFG = 1.0
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535

'''
Keep the server identity fixed so clients cannot connect to a different model by mistake.
固定模型服務識別碼，讓客製化 client 可以檢查是否連到正確模型；一般不應修改。
'''
REMOTE_SERVICE_ID = '06_Qwen_Image_2.1_圖片生成_Kaggle'
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整

'''
Limit concurrently parsed uploads to protect Kaggle RAM.
限制同時解析大型上傳檔案的數量，避免占滿主機 RAM。
'''
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2
REMOTE_STALL_HOURS = 2

'''
Choose where uploaded Qwen reference images are stored.
設定 Qwen Text／Edit／Multi 參考圖片的本機存放資料夾。
'''
REFERENCE_DIR = ROOT / 'qwen21_references'
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')


2. 安裝必要套件與 WanGP


In [ ]:
'''
Install packages required for the notebook interface and INT8 offload.
安裝 Notebook 互動介面、INT8 與動態 Offload 所需套件。
'''
%pip -q install "ipywidgets>=8,<9" "mmgp==3.8.2" "comfy-kitchen==0.2.35" "transformers==4.54.0" "diffusers==0.36.0" "gradio==5.50.0" "ffmpeg-python" "optimum-quanto" "accelerate" "einops" "hf_xet" "safetensors" "psutil"
!test -d {WAN_GP_DIR}/.git || git clone -q https://github.com/deepbeepmeep/WanGP.git {WAN_GP_DIR}
!git -C {WAN_GP_DIR} fetch -q --depth 1 origin 91301f0eda923b6c4b05159e32f1dfd2a960af9d

'''
Keep the source stable across repeated notebook runs.
固定原始碼版本，避免反覆執行時發生不預期變動。
'''
!git -C {WAN_GP_DIR} checkout -q --detach FETCH_HEAD
!git -C {WAN_GP_DIR} rev-parse --short HEAD
import subprocess
_WANGP_REVISION = subprocess.check_output(
    ['git', '-C', str(WAN_GP_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
assert _WANGP_REVISION == '91301f0eda923b6c4b05159e32f1dfd2a960af9d', 'WanGP 版本不符，請檢查 Git 安裝。'
print('✓ 第 2 節完成：套件與指定 WanGP 版本已準備完成。')


3. 模型精度與 GPU 設定檢查


In [ ]:
'''
Reject unsupported model modes before large downloads.
下載大型模型之前先確認模式有效。
'''
assert MODEL_MODE in ("int8_single", "bf16_single", "bf16_dual")
print(f"Selected mode: {MODEL_MODE}")
print('✓ 第 3 節完成：模型精度與 GPU 設定已選擇（尚未載入）。')


4. GPU 0 Transformer 預載預算


In [ ]:
'''
Ensure Cell 2 has supplied the GPU 0 memory budget.
確認 Cell 2 已設定 GPU 0 的記憶體預算。
'''
assert "GPU0_MODEL_GIB" in globals(), "請先執行 Cell 2，設定 GPU0_MODEL_GIB。"
import gc
import math
import torch
assert isinstance(GPU0_MODEL_GIB, (int, float)) and math.isfinite(GPU0_MODEL_GIB) and GPU0_MODEL_GIB > 0

'''
Check whether the existing model can be reprofiled in this session.
確認目前 Session 是否已經載入可重新設定的模型。
'''
state = globals().get("QWEN21_STATE")
globals()["GPU0_MODEL_GIB"] = float(GPU0_MODEL_GIB)
if not state:
    print(f"GPU 0 Transformer 預載預算已設定為 {GPU0_MODEL_GIB:.1f} GiB。")
    print("初始化模型後，若你再修改這個值，請重新執行本格即可套用。")
else:
    required = ("managed", "transformer", "dtype", "lora_path")
    assert all(key in state for key in required), "目前模型狀態不支援即時重新設定，請重新初始化模型。"

    previous_gib = float(state.get("gpu0_model_gib", GPU0_MODEL_GIB))
    state["manager"] = None
    gc.collect()
    torch.cuda.empty_cache()

    '''
    Reprofile existing model weights without downloading checkpoints or restarting.
    利用既有模型權重重新設定，不重新下載、不重新啟動 Kernel。
    '''
    try:
        updated = configure_gpu0_manager(
            state["managed"], state["transformer"],
            round(GPU0_MODEL_GIB * 1024), state["dtype"], state["lora_path"],
        )
    except Exception as error:
        try:
            state["manager"] = configure_gpu0_manager(
                state["managed"], state["transformer"],
                round(previous_gib * 1024), state["dtype"], state["lora_path"],
            )
        except Exception as recovery_error:
            raise RuntimeError("MMGP 更新及復原均失敗，請檢查 RAM 與模型狀態。") from recovery_error
        raise RuntimeError(f"套用失敗；已恢復 {previous_gib:.1f} GiB。") from error
    else:
        state["manager"] = updated
        state["gpu0_model_gib"] = float(GPU0_MODEL_GIB)
        manager = updated
        print(f"GPU 0 Transformer 預載預算已更新：{previous_gib:.1f} → {GPU0_MODEL_GIB:.1f} GiB")

    free_bytes, total_bytes = torch.cuda.mem_get_info(GPU_ID)
    print(f"GPU 0：模型預算 {state['gpu0_model_gib']:.1f} GiB；目前總占用 {(total_bytes-free_bytes)/1024**3:.2f} GiB")
    print("預算不是保證占用量；請在下一次生成時確認 VRAM 峰值。")

print('✓ 第 4 節完成：GPU 0 模型預算已設定或套用。')


5. 依所選模式下載模型


In [ ]:
'''
Import filesystem helpers before configuring the Hugging Face cache.
在設定 Hugging Face 快取之前，先載入檔案系統工具。
'''
import os
from pathlib import Path
import shutil
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"

'''
Import the downloader after the cache location has been configured.
設定好快取路徑後，再載入模型下載工具。
'''
from huggingface_hub import hf_hub_download

CKPT.mkdir(parents=True, exist_ok=True)
free_gib = shutil.disk_usage(CKPT).free / 1024**3
print(f"Models: {CKPT} | Available disk space: {free_gib:.1f} GiB")
TRANSFORMER_FILE = (
    "qwen_image_21_7B_int8_convrot.safetensors"
    if MODEL_MODE == "int8_single"
    else "qwen_image_21_7B_bf16.safetensors"
)

'''
Download the selected transformer, quantized encoder, VAE, and optional accelerator.
下載指定 Transformer、INT8 編碼器、VAE 與加速用 LoRA。
'''
MODEL_FILES = [
    ("DeepBeepMeep/Qwen_image_2", TRANSFORMER_FILE),
    ("DeepBeepMeep/Ideogram4", "Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_int8_convrot.safetensors"),
    ("DeepBeepMeep/Qwen_image_2", "qwen_image_21/qwen_image_21_vae.safetensors"),
    ("DeepBeepMeep/Qwen_image_2", "qwen_image_21/vae_config.json"),
    ("DeepBeepMeep/Qwen_image_2", "qwen_image_21/scheduler_config.json"),
    ("Viggle/Qwen-Image-2.1-viggle-turbo", "Qwen-Image-2.1-viggle-turbo-v0.2.1-6step-lora-r128.safetensors"),
]
ENCODER_CONFIGS = [
    "config_legacy.json", "tokenizer_legacy.json", "tokenizer_config_legacy.json",
    "preprocessor_config.json", "chat_template.jinja", "merges.txt", "vocab.json",
    "added_tokens.json", "special_tokens_map.json", "video_preprocessor_config.json",
]
MODEL_FILES.extend(("DeepBeepMeep/Ideogram4", f"Qwen3-VL-8B-Instruct/{name}") for name in ENCODER_CONFIGS)
for repo, filename in MODEL_FILES:
    print(f"Checking: {filename}", flush=True)
    hf_hub_download(repo_id=repo, filename=filename, local_dir=CKPT, cache_dir=os.environ["HF_HUB_CACHE"])

'''
Read the encoder's checkpoint metadata using the same helper as WanGP.
使用 WanGP 相同的工具檢查編碼器權重是否已包含視覺模型。
'''
from mmgp import quant_router
encoder_path = CKPT / "Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_int8_convrot.safetensors"
encoder_keys, _ = quant_router.load_metadata_state_dict(str(encoder_path))
if not any(key.startswith(("visual.", "model.visual.")) for key in encoder_keys):
    vision_file = "Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_vision_bf16.safetensors"
    print(f"Checking separate vision weights: {vision_file}", flush=True)
    hf_hub_download(repo_id="DeepBeepMeep/Ideogram4", filename=vision_file, local_dir=CKPT, cache_dir=os.environ["HF_HUB_CACHE"])

del encoder_keys
assert (CKPT / TRANSFORMER_FILE).is_file(), f"Missing: {TRANSFORMER_FILE}"
print(f"Ready: {TRANSFORMER_FILE}")
print('✓ 第 5 節完成：所選模型權重已下載並檢查。')


6. 模型初始化


In [ ]:
'''
Import standard utilities and PyTorch.
載入標準工具與 PyTorch。
'''
import os
import sys
import types
import torch
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
assert torch.cuda.is_available(), "請在 Kaggle Notebook Settings 開啟 GPU。"
assert torch.cuda.device_count() > GPU_ID, f"找不到 GPU {GPU_ID}。"
assert MODEL_MODE != "bf16_dual" or torch.cuda.device_count() >= 2, "bf16_dual 需要兩張 GPU。"

'''
Show a RAM warning before loading the large BF16 checkpoint.
載入大型 BF16 權重前，先顯示系統記憶體不足的風險。
'''
if MODEL_MODE != "int8_single":
    import psutil
    available_ram_gib = psutil.virtual_memory().available / 1024**3
    print(f"Available host RAM: {available_ram_gib:.1f} GiB")
    if available_ram_gib < 25:
        print("Warning: BF16 transformer + INT8 text encoder can exhaust Kaggle RAM. Consider int8_single.")

DEVICE = f"cuda:{GPU_ID}"
EXEC_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(GPU_ID)[0] >= 8 else torch.float16
existing = globals().get("QWEN21_STATE")
if existing and existing.get("pipeline") is not None:
    assert existing["mode"] == MODEL_MODE, "切換 MODEL_MODE 後請重新啟動 Kernel，再執行下載與初始化 Cell。"
    assert existing.get("manager") is not None, "MMGP 管理器已失效，不能直接沿用舊的 Pipeline。"
    pipeline = existing["pipeline"]
    manager = existing["manager"]
    parts = existing["parts"]
    print(f"Reusing {MODEL_MODE} model without reloading weights.")
else:
    torch.cuda.set_device(GPU_ID)
    sys.path.insert(0, str(WAN_GP_DIR))
    os.chdir(str(WAN_GP_DIR))

    '''
    Stub the optional rembg dependency because background removal is not used.
    未使用背景移除功能，因此建立輕量的 rembg 相容模組。
    '''
    if "rembg" not in sys.modules:
        rembg = types.ModuleType("rembg")
        def unavailable_rembg(*args, **kwargs):
            raise RuntimeError("本 Notebook 未啟用 rembg。")
        rembg.remove = rembg.new_session = unavailable_rembg
        sys.modules["rembg"] = rembg
    from mmgp import offload, quant_router
    from optimum.quanto.tensor.weights import qbytes
    from shared.utils import files_locator
    from shared.kernels import int8_backend, quanto_int8_inject
    from models.qwen21.qwen21_handler import family_handler, ENCODER
    quant_router.unregister_handler(".fp8_quanto_bridge")
    quant_router.register_handler("shared.qtypes.int8_convrot")
    files_locator.set_checkpoints_paths([str(CKPT)])
    int8_backend.configure("auto", 1)
    int8_backend._SCRATCH_BYTES = 384 * 1024 * 1024

    '''
    Keep the large token embedding table on the CPU.
    將大型 token embedding 表保留在 CPU，降低 GPU 記憶體用量。
    '''
    class CPUEmbedding(torch.nn.Module):
        def __init__(self, weight):
            super().__init__()
            self.table = [weight]
        def forward(self, input_ids):
            return torch.nn.functional.embedding(input_ids.cpu(), self.table[0]).to(input_ids.device)
    original_loader = offload.fast_load_transformers_model
    def load_with_dtype(*args, **kwargs):
        requested = args[0] if args else kwargs.get("model_path", "")
        is_bf16_transformer = str(requested).endswith("_7B_bf16.safetensors")
        kwargs.setdefault("default_dtype", torch.bfloat16 if is_bf16_transformer else EXEC_DTYPE)
        return original_loader(*args, **kwargs)
    offload.fast_load_transformers_model = load_with_dtype
    try:
        model_path = CKPT / TRANSFORMER_FILE
        assert model_path.exists(), f"缺少權重：{model_path}"
        model_def = family_handler.query_model_def("qwen_image_21_7B", {})
        pipeline, parts = family_handler.load_model(
            [str(model_path)], "qwen_image_21_7B", "qwen_image_21_7B", model_def,
            text_encoder_filename=str(CKPT / ENCODER / f"{ENCODER}_int8_convrot.safetensors"),
        )
    finally:
        offload.fast_load_transformers_model = original_loader
    transformer = parts["transformer"]
    text_encoder = parts["text_encoder"]
    vision_encoder = parts["vision_encoder"]
    vae = parts["vae"]
    text_encoder.embed_tokens = CPUEmbedding(text_encoder.embed_tokens.weight.data)
    transformer._convertWeightsFloatTo = EXEC_DTYPE
    text_encoder._convertWeightsFloatTo = EXEC_DTYPE
    vision_encoder._convertWeightsFloatTo = EXEC_DTYPE

    '''
    Keep VAE in FP32 to avoid FP16 numerical overflow.
    保留 VAE 的 FP32 精度，避免 FP16 數值溢位。
    '''

    vae._convertWeightsFloatTo = None
    if MODEL_MODE == "bf16_dual":
        pipeline.text_encoder.to("cuda:1")
        print("GPU 1: Qwen3-VL INT8 encoder placed on cuda:1.")
        managed = {"transformer": transformer, "vae": vae}
    else:
        managed = {
            "transformer": transformer, "text_encoder": text_encoder,
            "vision_encoder": vision_encoder, "vae": vae,
        }

    '''
    Configure MMGP from existing model objects; this is reused for live budget changes.
    使用既有模型物件建立 MMGP；後續變更預算時可重複使用，不重新讀取主模型權重。
    '''
    def configure_gpu0_manager(modules, generator, budget_mb, dtype, lora_file):
        '''
        Allocate the selected approximate VRAM budget for the generator.
        設定生成模型的預估 GPU 記憶體預算，並保留其它運算空間。
        '''
        configured = offload.profile(
            modules,
            profile_no=5,
            pinnedMemory=False,
            quantizeTransformer=False,
            extraModelsToQuantize=[],
            budgets={"transformer": budget_mb, "vae": 2048, "text_encoder": 2048, "vision_encoder": 1536},

            vram_safety_coefficient=0.95,
            loras=["transformer"],
            convertWeightsFloatTo=dtype,
            asyncTransfers=True,
        )
        try:
            loaded = offload.load_loras_into_model(
                generator, [str(lora_file)], [1.0], activate_all_loras=True,
                pinnedLora=False,
                preprocess_sd=lambda sd: generator.preprocess_loras("qwen_image_21_7B", sd),
                split_linear_modules_map=getattr(generator, "split_linear_modules_map", None),
            )
            assert loaded, "Turbo LoRA 載入失敗。"
            offload.activate_loras(generator, ["0"], [1.0])
        except Exception:
            configured.release()
            raise
        return configured
    lora_path = CKPT / "Qwen-Image-2.1-viggle-turbo-v0.2.1-6step-lora-r128.safetensors"
    manager = configure_gpu0_manager(
        managed, transformer, round(GPU0_MODEL_GIB * 1024),
        EXEC_DTYPE, lora_path,
    )
    offload.shared_state["_attention"] = "sdpa"
    original_encoder = pipeline._encode_conditioning
    def encode_prompt(prompts, images, device, batch_size):
        text_device = torch.device("cuda:1" if MODEL_MODE == "bf16_dual" else DEVICE)
        backend = int8_backend._backend
        original_forward = qbytes.WeightQBytesLinearFunction.__dict__["forward"]
        int8_backend._backend = "pytorch"
        qbytes.WeightQBytesLinearFunction.forward = staticmethod(
            quanto_int8_inject._default_quanto_qbytes_linear_forward
        )
        try:
            contexts = original_encoder(prompts, images, text_device, batch_size)
        finally:
            int8_backend._backend = backend
            qbytes.WeightQBytesLinearFunction.forward = original_forward
        converted = []
        for embeddings, mask, slots in contexts:
            if EXEC_DTYPE == torch.float16:
                peak = embeddings.float().abs().amax(dim=-1, keepdim=True)
                embeddings = embeddings.float() / (peak / 32768).clamp(min=1)
            converted.append((
                embeddings.to(device=DEVICE, dtype=EXEC_DTYPE),
                None if mask is None else mask.to(DEVICE),
                slots.to(DEVICE) if hasattr(slots, "to") else slots,
            ))
        return converted
    pipeline._encode_conditioning = encode_prompt

    '''
    Cache the initialized pipeline; later generations reuse the same objects.
    保留已建立的模型物件，後續生成不重新初始化。
    '''
    QWEN21_STATE = {
        "pipeline": pipeline, "manager": manager, "parts": parts,
        "mode": MODEL_MODE, "device": DEVICE, "dtype": EXEC_DTYPE,
        "managed": managed, "gpu0_model_gib": GPU0_MODEL_GIB,
        "lora_path": lora_path,
        "transformer": transformer,
    }

'''
Report the actual GPU configuration without claiming every weight is resident.
顯示實際使用的 GPU，避免將動態 Offload 誤稱為完整 GPU 常駐。
'''
print(f"Ready: {MODEL_MODE} | compute dtype={EXEC_DTYPE}")
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | allocated={torch.cuda.memory_allocated(i)/1024**3:.2f} GiB | free={free/1024**3:.2f} GiB")

print("Initialized model objects are kept. MMGP may dynamically move transformer blocks between RAM and VRAM.")
print('✓ 第 6 節完成：Qwen 模型已初始化。')


7. 參考圖片上傳與選取


In [ ]:
'''
Provide a multi-file upload control for the supported reference-image task.
為支援參考圖片的模式建立多選上傳介面。
'''
import ipywidgets as widgets
from pathlib import Path
from PIL import Image, ImageOps
from io import BytesIO
from uuid import uuid4
from html import escape
from IPython.display import display, clear_output
import shutil
_REF_ROOT = REFERENCE_DIR
_REF_ROOT.mkdir(parents=True, exist_ok=True)
_REF_MAX = 10
_REF_FILES = sorted(p for p in _REF_ROOT.iterdir() if p.is_file() and p.suffix.lower() in ('.png','.jpg','.jpeg','.webp'))
if len(_REF_FILES) > _REF_MAX or any(p.stem != f'{i:03d}' for i,p in enumerate(_REF_FILES,1)):
    raise RuntimeError('參考圖編號不連續；請清理舊資料夾再執行上傳 Cell。')

'''
Synchronize the numbered paths with the generation configuration.
將最新圖片清單同步到生成工作所使用的設定。
'''
def _ref_sync(paths):
    global REFERENCE_PATHS
    REFERENCE_PATHS = [str(path) for path in paths]
    pass

'''
Replace all reference images only after the new upload is fully validated.
新圖片全部通過驗證後才取代上一批；選得更少時也會清除多餘舊檔。
'''
def _ref_update(change):
    items = change['new']
    if not items:
        return
    if isinstance(items, dict):
        items = [dict(name=name, **item) for name, item in items.items()]
    stage = _REF_ROOT.with_name(_REF_ROOT.name + '_stage_' + uuid4().hex)
    stage.mkdir(parents=True)
    backup = _REF_ROOT.with_name(_REF_ROOT.name + '_backup_' + uuid4().hex)
    try:
        if not 1 <= len(items) <= _REF_MAX:
            raise ValueError(f'一次請選擇 1～{_REF_MAX} 張圖片。')
        for index, item in enumerate(items, 1):
            filename = Path(item['name']).name
            suffix = Path(filename).suffix.lower()
            if suffix not in ('.png','.jpg','.jpeg','.webp'):
                raise ValueError(f'不支援的圖片格式：{filename}')
            blob = bytes(item['content'])
            if len(blob) > 20 * 1024 * 1024:
                raise ValueError(f'單張圖片超過 20 MiB：{filename}')
            with Image.open(BytesIO(blob)) as photo:
                photo.verify()
            (stage / f'{index:03d}{suffix}').write_bytes(blob)
        _REF_ROOT.rename(backup)
        try:
            stage.rename(_REF_ROOT)
        except Exception:
            backup.rename(_REF_ROOT)
            raise
        shutil.rmtree(backup)
        current = sorted(_REF_ROOT.iterdir())
        _ref_sync(current)
        print('最新參考圖：', ', '.join(f'{p.name} = <Picture {i}>' for i,p in enumerate(current,1)))
    except Exception as error:
        shutil.rmtree(stage, ignore_errors=True)
        print('上傳未生效，保留上一批參考圖：', error)
    finally:
        _REF_PANEL.children = (_ref_new(),)

'''
Reset the browser picker so selecting the same filenames works again.
每次上傳後重建選檔元件，確保可以重新選相同檔名。
'''
def _ref_new():
    picker = widgets.FileUpload(accept='image/*', multiple=True, description='多選參考圖片')
    picker.observe(_ref_update, names='value')
    return picker

_REF_PANEL = widgets.VBox()
_REF_PANEL.children = (_ref_new(),)
_ref_sync(_REF_FILES)
display(_REF_PANEL)
print('目前圖片：', ', '.join(p.name for p in _REF_FILES) or '尚未上傳')
print('✓ 第 7 節完成：參考圖片上傳介面已顯示（不代表圖片已上傳）。')


8. T2I：文字生成圖片


In [ ]:
'''
Choose this task's image size; larger dimensions require more GPU memory.
設定本模式圖片寬高；較高解析度會增加 VRAM 使用量。
'''
WIDTH, HEIGHT = 640, 640
SEED = 42

'''
Select the number of image-generation sampling steps.
設定本模式的採樣步數，預設配合 Turbo 權重。
'''
STEPS = 6
TASK = 'text'  # 'text', 'edit', 'multi'

'''
Set the prompt for the selected task.
設定本模式的提示詞，修改後重新執行此 Cell 即可。
'''
PROMPT = '''
A cozy bookstore in Taipei on a rainy evening, warm cinematic lighting, realistic photography
'''.strip()
print('✓ T2I 模式設定完成：請執行 Generate 才會開始生成。')


9. Edit：單張圖片編輯


In [ ]:
'''
Choose this task's image size; larger dimensions require more GPU memory.
設定本模式圖片寬高；較高解析度會增加 VRAM 使用量。
'''
WIDTH, HEIGHT = 640, 640
SEED = 42

'''
Select the number of image-generation sampling steps.
設定本模式的採樣步數，預設配合 Turbo 權重。
'''
STEPS = 6
TASK = 'edit'  # 'text', 'edit', 'multi'

'''
Set the prompt for the selected task.
設定本模式的提示詞，修改後重新執行此 Cell 即可。
'''
PROMPT = '''
Transform the uploaded photo into a watercolor illustration while keeping the subject recognizable.
'''.strip()
print('✓ Edit 模式設定完成：請準備 1 張參考圖，再執行 Generate。')


10. Multi：多張參考圖片生成


In [ ]:
'''
Choose this task's image size; larger dimensions require more GPU memory.
設定本模式圖片寬高；較高解析度會增加 VRAM 使用量。
'''
WIDTH, HEIGHT = 640, 640
SEED = 42

'''
Select the number of image-generation sampling steps.
設定本模式的採樣步數，預設配合 Turbo 權重。
'''
STEPS = 6
TASK = 'multi'  # 'text', 'edit', 'multi'

'''
Set the prompt for the selected task.
設定本模式的提示詞，修改後重新執行此 Cell 即可。
'''
PROMPT = '''
Combine the subjects of the reference images into one cohesive cinematic scene.
'''.strip()
print('✓ Multi 模式設定完成：請準備 2～10 張參考圖，再執行 Generate。')


11. 生成圖片


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
assert globals().get('TASK') in ('text', 'edit', 'multi') and isinstance(globals().get('PROMPT'), str) and globals()['PROMPT'], '請先執行 Text、Edit 或 Multi 模式設定 Cell，並填入 PROMPT。'
with GENERATION_LOCK:
    import time
    from pathlib import Path
    import numpy as np
    import torch
    from PIL import Image, ImageOps
    import secrets
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

    '''
    Require a supported seed before building the inference workflow.
    在建立推論工作流程前，先檢查種子範圍。
    '''
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    print(f'本次生成 seed：{RUN_SEED}')
    assert "QWEN21_STATE" in globals(), "請先執行模型初始化 Cell。"
    assert QWEN21_STATE["mode"] == MODEL_MODE, "MODEL_MODE 已更改，請重新啟動 Kernel 後重新初始化。"
    assert QWEN21_STATE.get("manager") is not None, "MMGP 管理器未就緒，請重新執行模型初始化。"

    '''
    Ensure the requested budget was actually applied in Cell 3.
    確認指定的 GPU 0 預算已在 Cell 3 正式套用。
    '''
    assert GPU0_MODEL_GIB == QWEN21_STATE["gpu0_model_gib"], "GPU0_MODEL_GIB 已修改，請先重新執行 Cell 3。"
    pipeline = QWEN21_STATE.get("pipeline")
    assert pipeline is not None, "模型尚未初始化，請執行 Cell 5。"

    '''
    Keep every generation instead of overwriting earlier output files.
    自動編號保留每次生成的圖片，不覆蓋先前結果。
    '''
    def make_output_path(filename):
        base = ROOT / filename
        if not base.suffix:
            base = base.with_suffix(".png")
        if not base.exists():
            return base
        for index in range(2, 1000):
            candidate = base.with_name(f"{base.stem}_{index:03d}{base.suffix}")
            if not candidate.exists():
                return candidate
        raise RuntimeError("輸出編號已達上限。")
    assert TASK in ("text", "edit", "multi"), f"不支援的 TASK：{TASK}"
    assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0
    references = []

    '''
    Text mode never sends reference images, even when images remain selected.
    文生圖模式不傳參考圖，即使清單仍保留已選圖片。
    '''
    if TASK != "text":
        selected_paths = list(globals().get("REFERENCE_PATHS", []))
        valid_count = len(selected_paths) == 1 if TASK == "edit" else 2 <= len(selected_paths) <= 10
        assert valid_count, f"TASK={TASK} 的參考圖張數不符，目前選取 {len(selected_paths)} 張。"
        for number, filename in enumerate(selected_paths, start=1):
            path = Path(filename)
            assert path.is_file(), f"找不到參考圖：{path}"
            with Image.open(path) as source:
                image = ImageOps.exif_transpose(source)
                has_alpha = "A" in image.getbands() or "transparency" in image.info
                image = image.convert("RGBA" if has_alpha else "RGB")
            scale = (REFERENCE_SIZE ** 2 / (image.width * image.height)) ** 0.5
            size = tuple(max(32, round(length * scale / 32) * 32) for length in image.size)
            references.append(image.resize(size, Image.Resampling.LANCZOS))
            print(f"<image{number}> {path.name} → {size}")
    else:
        print("TASK=text：不使用參考圖片。")
    import math
    from mmgp import offload
    assert SAMPLER in ("auto", "turbo", "base"), "SAMPLER 必須是 auto、turbo 或 base。"
    assert isinstance(STEPS, int) and not isinstance(STEPS, bool) and STEPS > 0, "STEPS 必須是正整數。"
    assert isinstance(CFG, (int, float)) and math.isfinite(CFG) and CFG >= 1.0, "CFG 必須是至少 1.0 的有限數值。"
    use_turbo = (SAMPLER == "turbo") or (SAMPLER == "auto" and CFG == 1.0 and STEPS == 6)
    if use_turbo and (STEPS != 6 or CFG != 1.0):
        raise ValueError("Viggle Turbo v0.2.1 建議 STEPS=6、CFG=1.0；若要自訂，請改成 SAMPLER='base' 或 'auto'。")
    transformer = QWEN21_STATE["transformer"]
    offload.activate_loras(transformer, ["0"] if use_turbo else [], [1.0] if use_turbo else [])
    manager = QWEN21_STATE["manager"]
    if hasattr(manager, "refresh_loras"):
        manager.refresh_loras(transformer)

    '''
    Use the original Euler scheduler for the base model without the Turbo adapter.
    基礎模型使用原生 Euler 採樣器，不套用 Turbo adapter。
    '''
    sample_solver = "viggle_v02" if use_turbo else "default"
    print(f"Sampler: {'Viggle Turbo' if use_turbo else 'Base Euler'} | CFG={CFG} | Steps={STEPS}")
    if CFG == 1.0 and NEGATIVE_PROMPT:
        print("CFG=1.0：Negative Prompt 不會影響生成。")
    torch.cuda.reset_peak_memory_stats(GPU_ID)
    print(f"Generating: {MODEL_MODE} | {TASK} | {WIDTH}×{HEIGHT} | seed={RUN_SEED}", flush=True)
    start = time.time()
    with torch.inference_mode():
        result = pipeline.generate(
            input_prompt=PROMPT,
            n_prompt=NEGATIVE_PROMPT,
            seed=RUN_SEED,
            width=WIDTH,
            height=HEIGHT,
            sampling_steps=STEPS,
            sample_solver=sample_solver,
            guide_scale=CFG,
            batch_size=1,
            VAE_tile_size=(True, 256),
            loras_slists=None,
            input_ref_images=references or None,
            custom_settings={"qwen21_kv_cache": "Disabled", "rgba": "Disabled"},
        )
    assert result is not None, "模型沒有回傳圖片。"
    output = make_output_path(OUTPUT_NAME)
    array = result[:, 0].permute(1, 2, 0).contiguous().cpu().numpy()
    if array.dtype != np.uint8:
        array = np.clip(array, 0, 255).astype(np.uint8)
    image = Image.fromarray(array)
    image.save(output)
    free_bytes, total_bytes = torch.cuda.mem_get_info(GPU_ID)

    '''
    Show the requested budget alongside actual VRAM usage.
    對照指定預算與目前實際顯示記憶體使用量。
    '''
    print(f"GPU 0 budget: {QWEN21_STATE['gpu0_model_gib']:.1f} GiB | actual VRAM: {(total_bytes - free_bytes) / 1024**3:.2f} GiB")
    print(f"GPU 0 PyTorch allocation peak: {torch.cuda.max_memory_allocated(GPU_ID) / 1024**3:.2f} GiB")
    print(f"Saved: {output} | elapsed={time.time()-start:.1f}s")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i} allocated: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GiB")
    QWEN21_STATE["last_output"] = str(output)

'''
Check the real output before printing a success message.
顯示成功訊息前先確認輸出檔案存在且不是空檔。
'''
assert output.is_file() and output.stat().st_size > 0, '圖片輸出不存在或為空。'
print('✓ 第 11 節完成：圖片已生成並確認輸出檔案存在。')


12. 預覽與下載最新生成圖片


In [ ]:
'''
Read the latest output path and render the saved image.
讀取最新生成圖片路徑，直接顯示已儲存的圖片。
'''
from pathlib import Path
from IPython.display import Image, FileLink, display
assert "QWEN21_STATE" in globals() and QWEN21_STATE.get("last_output"), "尚未生成圖片，請先執行生成 Cell。"
output = Path(QWEN21_STATE["last_output"])

'''
Reject missing outputs rather than displaying stale notebook content.
若檔案不存在則停止，避免顯示過期的 Notebook 結果。
'''
assert output.is_file(), f"找不到生成圖片：{output}"
display(Image(filename=str(output)))
display(FileLink(str(output), result_html_prefix="下載原始圖片："))
print('✓ 第 12 節完成：圖片預覽與下載連結已顯示。')


13. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'image_multi'
REMOTE_FIELDS = ['task', 'prompt', 'negative_prompt', 'sampler', 'reference_size', 'width', 'height', 'steps', 'seed', 'cfg']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda}Z[s\x13\xd7\x96~\xd7\xaf\xd8\xa7\xab\xa6,QF\xd8&0\x13*J\x95\x03\nq\x85\x8b\xc76\x9c\xa9C\xb9T-iK\xee\xa1\xd5\xadt\xb7l<\x1eW\xd9$\xc6r\x8c1\'1&\x18\'\x0eI \xe6\xe2\x0b\x90\x04c\x0c<\xcc?\x99\xb8[\xd2S\xfe\xc2|k\xef\xeeVK\x98\xa1\nI\xdd\xfb\xb6\xee\xeb[k\xbb\xa3\xa3#\xd6W*\x9b\x96\xc3\xac\x8a\xe1h%\xce\xf2\x9a]\xd6\xd5q\xa6\x1ay\xa6\x95\xd4"g9\xd3\x18\xe5\x96\xad\x99\x06sLS\xb7\x93\xb1\xfa\xde\x9c;s\xbf\xb6\xb4\xe6Uo6\xa6\x9e\xb9\xbb\x0f\xfe\x9c\x9avW\x97ks\xb3\xf5\xd7s\xde\xe2\xddzu\xb6\xf1\xe3\xcd\xfa\x83\xd7\xee\x8b\xfb\xee\xcc\x8b?\xa7\xae\xc6:p\x94&\x8f\xa2sb\x05\xcb,\xb1\xb2\xea\x8c\xe8Z\x96\xf9\x03\xfdx\x0c&\x19\x95R\x19T\xd8\xcc(\x87\xebL+7"\x17\xf6\xf7\x9d\t\x16\xf5\x11\x91\x9d\xf2\xeb|\xd9\x8e\x89\x93Ns\x83[\xaa\xc3\x99\xca\n\x16\xb7G\xd8\xd1\x9e\xc3Y\xcda6\xe7yf\x1a\xfa8\x1b\x1b\xe1\x06;\xdc\xcd4\x9bY\xfc\x8b\n\xb7\x1d\x9eO\xc6\xdc\xc5G\xde\xea\\}}\xc3\xdd\\\xa1Q\xef\xceUon\xc1}\xb5[{<\xef-o\xd7V\xbe\nvj\xac\xac{\x0f\xd7j\xeb\x9b\xee\xc6\xcd6\xfel\x9e\xb3\xb8\xe3S2\xc0mS\x1f\xe58\x94\xcb\xd3\xcb\xdc"a\xb31\xcd\x191+\x0e3!\xdc1Ks4\xa3\xc8\x9c\x11\xce\x0c\xd3\xe1Y\xd3\xbc\x8c\xd9\x0e\xbdL\xc6\xbc\xadE\xef\xc9=)n\xef\xe9.H\xdb\xdf\x99r\xa7\xe6\xe5\xe1\x7f\xed]\xdf\x7f}m\x7fg\xa1\xfe\xe0\x9a\xbb\xf5\x98\x9d\x0b\xd6\x83X\xf7\xc6\x9a\xfb\xeb\xbcd\' r\xe0\xc2\xb9\xcc`:}\x8a\xa5\x02B\x93\x16\x94\x9d\xe5\xba9\x16\xef9t\xe8hO\x82i\x05&\xa7\xa4H\x08\\\xb79\xd3\x0c\'N\xef\x12\x01[_T4\x8b\xe4kW\xca\xc458\x13\xecey\xc1\xc4\xfblE\xd3\xf3\x01K\x9aQ\xe0\x167r\x9c\x8d\x99\xd6\xe5\x02\x0e\x82\xa4W\xd7}\xb1\xdeX\xaf?^\x86\xa1\xec\xbf^\xf5~\x9f\xae\xad\xcf\xbbs\x0b`\xca\x9d\xa9z\x8f~\xf2\xd6\xeeK6k[_\xb9\xab\x0b\x01\x13\xaamsH\xba\x8b}\x94b!C\x1f1A~\'\xeb\x10\x8f\xee\xdb\x99\xc6\x8f\xd5\xfdWP\xdc\x1ef\xd6g\x9f\xb3\x0fz>\xfc\xe0\xc3\xe3\xff\xda\xf3\xe11\x1c\xe0U\x97!\x99\xda\xd5]\xf0\x88};$c\x83#\xe6\x98 Z\xcd9\x15U\x97L\xd9&\x18%)\xc1\xf4Hw9\xd5\x00\xa30\x9c\xb2e\xe6+9\xcc\xd0alV2\xd6\xb8\xb7U\xfbe\xd7\xddz\xd0X\xb9\xbb\xff\xfammi\x1dZ\x08\xf5\xe4-\xbf\xdc\x7f\xf3v\xff\xe5\xbc\xfb\xe6zcv\xa1\xb6\xf8\xc6\xb7"\xa1\xda\xda\xef7\xbd\x1fV\x03\x0e\xcb\x16I\xbc\xd0\xe1\xad>\t\x95/\x88\xf9koe"`y\xb2\xc3W\xc7Eni\x85q\xd0\xad:\x82\xf8\xd0\xa4Y\xc9\xccsVR\x9d\xdc\x08\xb7%_\xba\xc5\xd5\xfc8\xd3M5\xef\x8f\xeb\xc9X\xed\xa7\xdd\xfa\xa3\x05\xdf\xc6\xd6\xef\xb9{\x8b\xf0a\xf7\xc53\xe9\xea\xf4\xe6\x87yX]}\xf67\xf0\xd1\x98~\xeb\xce,\xc0\xe4\xdc\xedE\xf7\xed\x97\x92\xd1z\xf5k\xef\xc9\xd7`\xabMG\xca\xbf\xff=}\xae\xa7;38\xd4;\x94V`\x0b\xac\xa8\x9bYU\xb7\xe3\x89N\xa6\xd4\x1f\xcfC\xd1\xee\xda\x8b\xfa\xbd\xeb\xf2\x14\xb7\xfa=l\xd6\xbd\xbe\xccNr]\xc7fJ\xb0St\xa3K\n\x11\xae\x0c\x93\x89\x9e=\x7f*}&C\x9f\xd8\xb0\xf9\xc0@\xbew\xf77o\xe9%H\xc69 \r.\xec\xdeZs\xe7o\xb1\xcf\xb9ep\x9dIE\xd0\xeb\xe0Tq\xa0o\xe2\xff\xc9s\x0e\x02!h\x1eUu-\xcf\xce\x9e=\xdd\x0fY\x1a\x884V`\xe8\xb6\xa3Z\xc2w1\x8f_)s\xc3\xd6\xe0\xedE\x19\x7f\x103\x93\xb1\xd3\xfd\x17X}\xfd;\xef\xda\x1f\x8d\xc7K\xb5\xcd{\xb5\x9b\xd7\xdc;\xeb\xb5\xaf\xeey\xb7\xaa\x88/d\xea\xab\xeb\xf5\xa9\xdb\xf5g\xaf|\xf9\xdf\xb9\xdaX\xfe\x96\xcc\x04\xa1gz\xd5\xdb\xf8\xa9M\xa4QA$\x8b\xdc\x89+>QJ\x82\xe2\x19"\x08\xa2\x80\xc1I\x1aDqx\xa6\xb7\xfa\xc8\xdd~Z{\xf1M\x8b@\x0e\x92}S\ni\xc3\xae\x80\xcdV\xa3\xcaV\xf28\x96\x8d!<K7A@U\xcbe]\xc3\x18\x14L\x8acG\x03\xa3\xf2\xae\xcf"\xfcP\xe8$It1d\x86\xda\xe6m(\x07l\xfbS\x99\xb7\xf13L\xce\xbd\x7f\x1b\x86\xd4\xc6,\x16ue\xa4VO\xf7}B\xfan\xb5\x83b\xb9\xd2\x95\x11V\x9c)jYe\x18\\\xb7-\xc1Q\xfbo7C;\x80\xbdE9\xf7I\x10\x1c\x97\xb52\xd75D\xea\xd4\x012\x0e\x06\x95D@Z8\xbdU\xe8\xbe,\xb7WH\xde\x81D\xfd\xb3#G\x1ek\n\xf9s\xce\xcb\x8c#\x11\x8cG,\x07\x92\x84\xb0U\xe4\xabBK\x92\xe0\xaa\x05A[\x0c\xc9\xa3\x8c\xfcQ\xd0tN9y\xf6\x11\xec\xba\xf6b\xbd~gm\xff\xed\xf7\xb5[w\xa2i\x83r\x81\xc8\xcf\x94-D\xaa\xa8\x7f+\x1con\xa15\xec\xe4y\x01\x16~\x99g\xe4\xee\x19\xca\xd0q:\xc2PK<q"\xc6\xf0/\xab\xda$!\xca\xd6q\xe5\xc8e\xb5X\xd4\xf9\x11\x8a\xec\xa0\x0e\x16x\x84\x05\xf3\xc5l\xe4\x12\x92\r-J\xda\x95BA\xbb"w\x89\xec$\xc6(\x1ff\xe4\x84\xb8\x92,\xd3V\xef\xac\xe7W4\xdbA\xe0h\xee\x80\xfcU\xb1\x0c1*\xde\xc1\'!\xb7<\xbfBv\x88\x88]\xe4q$\x84\xee\xae\xae\xae\xc8"\x84\xef\xbc\x96\'x\x10=\x9b(\x8e\x17\x94\tI\xa9\xc3K\x93\x99\t\xb1\xd5\x89\xae\xa3\xf9\xc9\x89\x08\x03\x93>i\x11\xf2\xc2-\x0f\xa01Bg8M\x0cZ\xaa\x06\xfe\x07$\xf2J[\x96i\xc5\x95\xfa\xde\x8e;\xbb+\xf5\x08\xc3mL/\xed\xef|\xdd\xb8C\x18C\t\x82=\x05#\xa2\x9e\x9c\x121G\xd7r\x9a\x03\x0f\xb4\xb9\x8e\x80\x05\x17tT\xfbr\x10\x9f\xcaHR\xaaEvcq?\x0b\xc3Z\xc2\xe4\xea\xc7\xee\xa9\x99\xd0Ke\xf4\xa7\xa8tm\xa1~\xe7\x16E\x8e\xc5/\xebS_J\xfbis\xce\xa1\xde\xc1\xcfI\xd2q\xc5\xe1W\x1c\x05\xc6\xcf\xf3\x9a\xf8.UtGS\x10\xe2\x0b\n\x0c\xce[\xda\xf2\x16\x7f\xa3\x18@+(\x85\xd1\xf7\xa4\xd2\x0e&\xca\xa6\r#\'\xb8$\x8d\xdb\xd6\xfe\x8b\x00h\tf\xa8eu.@\x93`\xbb%mI\x99\xd5\x7f\xfd\xd9\xfb\xe1&\xc0(R:\xe2\x89wkG\x18\xfbR\xed\xc9\x03\xf7fU:%\xf9\xc16\xf2\xf3N\xfd\xc1\xb4\xf7\xf4j\x1b7\x7f\xef;5\xf4\x19\xfb\x181\x8a\xd0\xefg\xe9\xbe\xd3\x9f\r\x85\x8fr\xf0_\x80\xff(\x06\xb5L\t_\x06\xec\x08\xf4\x13\x11\xb80\xcbw\xfc\xdb\x01\xaa\x185\x91W\x90C\xc0\x9b@\xdbv+\xdc\x93\x81\xaa\xbe9\xe5..\xd7\xeen\xc2]\x1b\xd3;\xf4;\xe2\xcf2\x1b{\xcfH\x91\xb0\x15\xca\xd2{\x8fB\x9d\x05,F\x88I\xb1K\xc3\x92\xd2!(M"\x04\x83\xa8\x83\x05\x19y\xbbI\xb7OR\'\x91nH\xc4,\xdf`JI\x85\xda\x03\x8b\x03\xd1\xcb\xb3\xa0\x18\xe7I\xf3!lp\xf5yH\x04Y\xd3\xc2s\xd8\x9a\xb73\xe3.o\xee\xbfZ\x90!J\x18\xf8N\xabe\xc1\xa1\x84U\xfd-\xc5\xa4Q\x9d\x88\tg\xa11\xfa\x1e\xa0\x90H& \x0f\'A\x12-\x0e\xfeSf\x1a\xf7\x11g\x9e#\x17\xe7\xb9\xe1\xb0J\x99\xa0\x0e\xcb!\xe6&\xc5\x0e\xbe8\x17\xd6\x81>\xe5\xf12 {sS\xb5?6\xea\x1b/\xc8\x0bV\xa7(C\x08\xca$\xd1D_\x94\x8e\x80w\x11"I\xa8:\xfc>\x1eb\x1b\x993\x06\xd2\x9f\xa6\x07\xd2\xe7N\xa63\xfd\xbdC\x9f\r\xc21.\r\'\x12\xad\x0c\xa5\rXG\x8e<\x19\xb9\x14nL\x95BS\x05\xc2r\xe0S\xc2\xde\x9c1\x93\xe1\xd9\x19\xb1\xb8\x1c\x11^&\xb9\x02\x89D.\x02\xc7\xde\x16\xe2\x85_%\xec\xfdN\xb2\xffe\x05CR\x1br\xc8\x9dy\x08<\xbc\xbf3\x87\t\xed\x8c\t\xac\x93\xc9\x99\x08K\xc4\x157\xe2\xad\xac&\xc8\xd4\xbbY\xa0\'<H\xa7\x97uB\x0fA\xf2\x83\x16\xe1uw\x978\xc0\xf7\xb6\xc89\x14%h\xb3\x94\x0c\x0b\xa2t\tl\x07\x14\xc2\x95aP\xf0c\xf0\x12\xf5\x026q\xc0A\x93L\xf2\xa4\x1c`6$C\x94\x83\x14T\xc2`)\xf2gX\x8d\xe5F\x903(Z\xda\xa86\xa9V\xb44<\xaaT\xf56Mg\x7f\xe7Am\x07\x80\xe2M\xd4~ef\x05\xe0D\x1a\x96u\x17\x9c\x11N\xe0=Z\xf2\xeeU\xdb\x85L\xcaC\x89\x9b\xe5Vg\x98.)\x90r\xbc\x14\x95k\x1b[\x9d\x12h\xa6\xba\x13\xbe;D7\xa3\x7f\'Gx\xee\xb2\xb0}\xda.,\xc38\xa87\x04j\x08\xb1iX\x8d%\xc3\xc5!\x00\x15\x10MVd>\xfe\x9c\xa9\xca\x18\xdb\xd4\xc7\xc6w\x98\x1e\xf0\xd3N\x06\x11\x1b`\x83\x106\x84\xa3\x01n\xc2hR\xb334!.R\x847\xf7\x86BFu;\x122V&h\xde\xa4r0\xbf\xbd\x80\x9b\xe3\xa4\x1e\xb8\xb8\x0c\xa7\xa4\\d;\x9c\x00&\x1d\xe4~\x1b\x89\x8f\x02@{<k\xf2M\xb0p\xe3g\x96\xfe\x8f\xbeO\x19\x8a3\xf7\xe6?\x85\x16\x1f\xc8\xe8\xd4\x98Z\xf4\xbe\xbb\xd1\xcc~\xc8^\xaa^\x1eQYcj\xa51\xfd\xed\xfbD r\x94hG$M\xc8<N\\$\xa8\x93a\x9b\x15\xf8y+.\x90-\x96T\xd8\xbe \xf8P\xc8H\xeaM\x1bV \xd6$Z\xd6\x8c\xa8vF\x12\x02\xd7\xeb\x15%\x95\xd8\x86BN\x16R@\xf4\xa1(\xa14e\x90\x1b\x8f\xcc\x82\xfe\xcd\x03i\x90\xc3\xb2\xddC\xb1\xeb\xf4\'\xb4y!r\x9e\xf0p\x1aP\x12\x07k\xa5?\x90\xbfj\x97\xa9x\x12\x99\x8eUl\xb2\xc0f,\x1c\xec\xfbG\x9a\x04\x82\x92\t\xd4\xa9\xc0+\x16H\x8f\xa8\xe5\xd5\xfd\xf6\xd9\x14\x11^>\xa7\x02\xfa\xf5*\xf2{\xe3\xfb\x9fj\x0f\x17\x10\x0b\xbc\xf5;\xa42\x99M\x84\xd75n\xbdp\xb7\x9ex[K\xef\xd3\x8f\x9d\xa3l\x9bb\xf1\xb6#\x0e\x1dB\xf0:\xc2\xe2R\x0ccZ\x1ej<\xe4\x0be\x84k\xc5\x11\'\x91\xa0I]\xc9c\xcd\xbd(\x94\xa4\x98\x83\x1c\xc3\xe3%\xf5J\x9c\xda\x0f\x16BZ>\x0e\xdb/\x8a\x1d\xe4yG\x185V\x0e\x89Or~\x7f8\xd4\n\xed\x94\x88\xc0\xda\x10\xab\xa1\xaeB"\xf3\xa9\x92\xc1+N\x1f~\xcb+9\xc0m\xb5\x04\x0ch\x14\x93gz\xcf\x9d\xfc\xc7\xf9\xc1Ds\x1f\xbf\x8b\xa0|$\x96O\xc8\x883\xf91\x13\xae\x95$\xe7\x9cd\xff{\xed\x1b6A;\x12\xac%\r\xb7%\xdc\xb3(\x07\x18\x12P\x804e\x97A5\xc6\xc9\xdbF5\xb3bG\x91\xce\x98\x86L\x9a\xc5\x82\xa2A\x01T*\x15n\x84("\x9b#!R\xa0h\xb9\xea7\x0f\x10i\x80y\x9a\xd8&\x8a^\xa2\xb4H~d\xae p\x80(\x81m\xfc=Za\xea\x81h9\x10\x1555\x1d\xcb\xd4\xed PZ\x15\x19&#\xc1\xd1oL!\xd9<\xbc]\xdf\x98\xf3n\xdc\xf3\xd6\x7f\xc6!\x12XJ\xd4\xe5W\xf1\xb7v\xdc\xa9\xbd\xb6.`\x89\x9a\x99\xa2YY*\x15\xcbA\xb7\xd2,\x14\x08\x8d\x04\x88s\xb0\xf7l\xff\x99\xf4\x80\x84\xd0j\xc51\t:\xa3R\xc8\x8a\x1fTr\x10\x84V\x82i\xb2\xa5\xe5}\xb7\xc5h\xee\x9fS\xd3b*\xf3\xaa\xcb\xa2\x8e\x896K4\x9b\xcaG\x15\x9c\xc4\x07\x87\xd2\xfd\x83\x9d\xd4\xc1K\x88 I\xf5\xca\xbb\xc3Y\xd3\xd4\xe5\xb8xA\xb8\x97N\x16\xbf\xc3s\x05\xae\xfe\r\xfc\xbe\xe7\xa8\x93\x9f\x9e\xee\x84\x07\x89\x8c\x0e>\xc9ch\xc7\x92\x0c\xf8\x05\xcd\xd0\x1c1I\xbe\xc6\x0f\xf61`A\x92N\xa2\x87\xf0\x1c\x80\x13w\xfb\x9f4\xc2\x04\x18\x9b\x03l\t\xa5\xecW\r\xe7\xa9\x8dKJ=~\x18\xd5Z\xb9\x93\xb6KuS\xb5\x90E\xc6\x16)\xa1b\xfbM\xafQ\xdf\x08\xf2\xec\xa2F\xe5*\x1b\x12\x82\xb3x\x0e%|\xd0\xf6ug6\xbc\x8d\xfb\xa8\x18\xfc\x9dds\xc6\x9b\x0b\xec\x0b\xc9^X\x02%\x82\x96m\x1a\x80\xdf\xcb/\x03\xfd\xe3\xd0\x8c\xd4\x0bbL\xa09BIR\xaf"6\xb7\xbc\x17z\x0f\x05\x92\x12\x02\x89(\x02/\x8e\'\x08\x1677\xa6A\xa97\x82\xc9\xc7iGZ\xfa7\xb1\xd4\xaf<eq\t\xeb\xaf\x04\xa5e\x0b\xcd\xa3]\xc9\x9ed7s_\xed\xd67\xb6\xe5I\xa9\xe3\xb0(\xc1z\xb2\xeb\xaf\xbd\xbb\xf5\xf9\xfb\xa8\x92\xea\xb3\x8f\xea\xebWe\xf7\xc2[zI]H\x9f\xf6T\x07\xd9\\\x87\xb0\xbe\x0e\xe2\xa1#\xe2u\xbd@\xe6\xa3\xcd\x1a\x15\xd0\x98\xbcK\x1e}\xc6\x1c\xe8\x95MxQ\xad;\xb6\xecO\n\xd4E\xfeI\x8dT\xa9\x11\x04\x04\xef\xdbYw{\xc1\x9b\x15R\x97\xeb}O\xbc\xb3\x1e*\xc8\xbd\xb5&\x15\xe4\xdfL`*\x1d\x12hD$B\x9c\x05d\xc5\xda\xdbG\x911e8\xe6\xbbgR\xf5\xc9\xcf\xe8\xa6\xa5\xda\xf1\xc8$ x\xa5K\x19f-\xfa\x10y\xf1\xd20\xc6 \xba\x83\xc7\xc2\xbe\xb9\xbc\x8f\xa0\xf0I\xc1\\T(B Y\xdd\xcc]FN,8\xa0\xd21\xa1*\x12H\x80K-Nt\x05\ru\xbf\xfau\xab\xb3\xde\xe2]\xb9\xdc}s\xdd\xbdy\x1d\xf6K\x8d\xb5\x9d\x9d\xc6\xaf\xa2\xe7\xb30\xe5\xde\xfbZ"SYJ\xee\xef\xccCF\xd4\xfa\x16\x92\xda\xdfy%\x0b\xe3@TAw\xb3]LA\x83q8&\x01\x81\xea8V\xdc\x7f\t\xe7\xb5$[R\\\x8ao\x81\xfep\xb2e0*K_&\x17li&\x01\xd6f\xe9\x8aN\xc5hn\x84\xe7\xc5\xaf\x82\xa8y\xb8l\x17\t\xdeC\xb9\xd0ki\x15j^-\x8b\x1e\xbc\xbb\xb6[\xfb\xe5\x86\xdf\xb2\x96\x9e{\x83Z\xec\xfe\xb6\xa1\xf5H\xb1\xc8\x9ec\xeb\x1e\x810\xa45f\xc4\x05\x0e\x89D\x19\x15\x1e\x94\x19\xed\xeaQ\x0eP\xb2\x92\xe7\x05\x15\x05\x99\x124\xef\x95Ai\xce\'\xd8DG\xd4\xf9:\x0eX\xdc\xf1\t1\'H\xec\x98d\xff-B\xd0\x04>\xe8\xf7 \xc2\x9b\x9d\x9a\x10NJI\x1a\xab\xdb"\xc5\xb9\xf4\xe9\xde\xa1\xbe\x8b(3\x07\xce\x9f\xed\x1f:\x11M\x96\xa1G\xaf\x9c\xe3EUtW\xfa\x91\x96\xca\x0e\x93\x19\xd8}\xfd\xb4\xb1\xf6V\xf6\x1b"\x1e\x0c\\\xc1\x01\xd5\xb9z\x99J\x0f\x87<8G\x18\x16"\xa7;\xb3\x8aA\xd7\x1fHi\xb6\xf0`s\xcc`\x17\x07z\xcf\x82-\x02%1\xba\xdex\xbd\xe9w|\xdd\xe7\xdb\x88\xdc\xb5\xdf\x9f\xd6\xd7\xabT\x8e."\xa4Tk\x9b\xb7\xa3w\x1b\xe4\xb3\xfd\xe3Ct\xc5\xd7\xec\x96\xcb\x85\xa1\x1b\xd3`2W\xc9\xab\x84\x83\xb8\x93!\xe22%^2\xad\xf1\x0c\xd1h\xc7q^\xa6/\xb8\xa3J_\xe1\xb9\x8a\x1f\x82\xc8G\x0e\xab\xb9\x1c\x8a*K\xe4\x01?\xd2\xf8\xd8\xb4%J\x05m]p1\xb5\xd8\x98\xbeA\x1e\xf5\xf6\xb1D&\xac?\xe8\xf9\xba\xdf\\\x8f\xc63\xc9Ek+#0\x03\xff6\x12{\xc3\x12\x9a\x17\x14\xa4Z\xbf\xe0\xc5\x0f\xd1_\x9a\xfc\x9f\xdb\x13\xb2\xa9D\xef\xe8\xba\'\xd5\xbc\xecQ(\xa9V\xec\x91\xd4\x90U\x01V\x14\x05!\x81O\xad\xc4\x93\xf4\x11O\xc4d\x7fL\xc8)D2\xa29\x1e\xf4#!8X(V\x85L\xfa\r)\x1eo\xb64\r\xd1\xf9\x156\x92\x92\x06\xd5\x19\x0e\x1a\xc1@\x9b\xc95g\x08\xaa\x03\xa2;#e\x11\x00uJp\xd9|)quJ\xb2\x1c\xd9\xc2\x87i\x19[X\xbeD(\xad\xa3\x81[\xa6Z\x9e\x9a\x93\x8a\x15\xc4\xd6\x8c\xc0\xde)B%\x91\x9e3rM\x86\xf0n\xaa\xbb\xf9\xf6bo:\xe3h\xb4\x0f\r\xc4I\xc2\x9d\xac\xe7\xd8\xf1Ds\x8a\x08_\x19\x9b\xda<vJ\xb4\xfa\xdb$\x868\x97\x91\x15f*\x02\x88\x11\xbbZ\'\xe7*\xb6c\x962\xfe\xf5.\xfcZ\xf9b\x8c\x1b\x08\xb5\x97G39\x15AO9\xc1\x94S\x9a\xadfu\x9e\'\x18h\x15\xb3j\xcb\xbbI\xb9\x99o\xe6\x83\xea\xa8\xb4\xf1@\x93\xc1\xc5}\x10#\xfd+}\x01o\x1d\x82\x9a\x91\xc9\x84\x92dg\xcc\xfdr\xc6\xfd\xea\x19\xaa\xfb\xa8%\xfb1ru\xdd\xbf\x80\x14}2\x89\xe3\xdbz\xa7\xbea\x1dt\x13\xe2=\xfb\x86\xd0\xd5\xdd\x1f\xa8%\x18\xc2\xf3\x98\xdf\xe2M\xbd{\xdfp\xfe\xc2P\xff\x85\xa1\xcc\xb9\xde\xb3\xe9DL\xb5,u\x1c\xb3\xe4\x01\x97Nt\xb2\xae\xe1d\x99[%\xb8v\xbc\x1bJ\xc2\x8b\x04U\xad\x8eV\xac\xa0\x18\x89\xe3\xa1\\\xc1\xa7\xf8\x0b\x82\xb8\x88\x96b\x8fd\xde\x19/s\x02IF9Y\x81g\xfe\x9bt\x89\xe0\x00\xbc\xcd\xe9Z9.\x9e;\t\xfd\xf6\x1c;\x96H\xaa6-\x8b\x07k\xb0_\xb4dO\x12\xc2\x17+\xe4:\x7f8iC)q\xc9S\x18O\xe5\x1f1@\xf6\'/\x9c\xeae\xaa\x8e\x84\xaf\xc2We\x8c\xa5(\xee_9\xe7QU\xe5\x90\xe8D`cf.W)\x03\\\x8f\x07\xb7\xcbry3FV\xaf\x01\x7f"\x9f\xc9`Y\xaf\xce\x8a\xa8+\xef\xa0\x9b\xb3\x16~\x8c\\\xad!\x1f\xf3Lv\xdc\xa1\xde\xaec:\xaa.\x1f(\x984c,\xce\xcf\xa0,\xcfP\xbf\xa05\xb0\x86w\xe4\xef\xdc\x08\xaa\xba\t\x93\x86\xe7\x05\xbcDs\x02p\\m\xe6Wy\xd1 /\x01A\xac\xec\xe7Ij%\x7f!\xcd2y7f\x17\xdf\x89\xa7"\xab\xc8#\x11Q[\x90JG\xeb}`\xc7\xf0\x89dwa\x92\x9d\xd6>A@\x8d\x10\x85u\xf1(\xeb\x87YS(t\x87\xd5\xdd\xd5\xf3\xc1\xa1CGO${\xe4j\xe5]5\x06\xe9\x8a\xc8\xf1\xb5I\xee$\xd4)!\x0b\xbc!z\x13\xec\x17\xc0\xef\xc9zm\xdc\xfbzm\xc9\x80\xad\x02\x08\x16\xb6\x9d\r\xce\xa2ZT\xaf\x04)\xd2\x9f\xc7\xf3\x816\xff_6\xa9\xa1\xde\x04\x9b\xcc\xcc\xd2m\xb8\xed\xb7Pe\x85+\r\x14@F\xb0\x8b\xed\xd9(jA\xba\xa7\xc1\x01v2\xe6\xdfC\xca\x9b\x97\xb9\x87\xfb\xaf\xfe\xd8\xdfy\xe0Vo\xbb\xb3\xbb\xde\xdc\x14b\x82\x10\x1dd\xd0\xc6\xfa\xc1Z\xa78\x97\x07s\xd2\xab(?r]-\xdb\x94"#)\xf0\xb0H\x8bB\xe9\x00\xa21Qa4\xef\x02#\x82\x91^&\xfb\xcf\xf1D"\n\x99\xa4\x80\'\xb4I\x16\x8a\xacM\xa8\xed\x02\xd5\xde#\xcbV\x10\xad#\x96\xf8\x81\x8e\xfer\x01\xb8\xca\nb\xc4\xff\x01p\xb4nc').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['text', 'edit', 'multi']
REMOTE_MAX_IMAGES = 10

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'Kaggle CUDA GPU 不可用；請啟用 GPU 並重新執行模型初始化 Cell。', []
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [0]
        else:
            selected_gpus = [0, 1]
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != 'cuda:0':
                return False, 'Qwen Transformer 沒有設定為 cuda:0。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, 0), ('HeartCodec', pipeline.codec, 1)):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐 GPU 記憶體）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        matches = [int(number) for number in re.findall(r'<Picture\s+(\d+)>', str(fields['prompt']), re.I)]
        if any(number < 1 or number > int(fields.get('_image_count', 0)) for number in matches):
            raise ValueError('R2V Prompt 的 <Picture N> 編號超出已上傳的圖片數量')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files)))
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 13 節完成：選用 FIFO／ngrok API 已啟動。')


14. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_Qwen_Image_2.1_圖片生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 14 節完成：API Schema 和 client 說明已顯示。')
